# 1# Setting Up

In [1]:
#Importing Some Functions
from cdb.sympy.calculus import diff
from cdb.core.manip import eliminate_kronecker

#Spacetime Indices
{\mu,\nu,\rho,\sigma,\alpha,\beta,\gamma,\kappa,\chi,\theta,\phi,\xi,a,b,c,f,h}::Indices(name=spacetime, position=independent).

#Dimension
d::Symbol.
e::Symbol.
n::Symbol.
\tau::Symbol.

#Coordinates
x::Coordinate.
y::Coordinate.
t::Coordinate.
r::Coordinate.
u::Coordinate.
v::Coordinate.

#Order counting
x^{#}::Weight(label=order,value=1).
y^{#}::Weight(label=order,value=1).
r::Weight(label=order,value=1).
t::Weight(label=order,value=1).
\tau::Weight(label=order,value=1).
u::Weight(label=order,value=1).
v::Weight(label=order,value=1).

#Projector
h^{\mu\nu}::Symmetric.

#SolidAngle of (d-2) space
\Omega::Symbol.

#Zeroth Metric
\eta_{\mu\nu}::Symmetric.
\eta^{\mu\nu}::Symmetric.

#Metric
g_{\mu\nu}::Metric.
g^{\mu\nu}::InverseMetric.

#Kronecker Delta
\delta{#}::KroneckerDelta.

# Partial Derivative
\partial{#}::PartialDerivative.
y::Depends(\partial{#}).

#Covariant Derivative
\nabla{#}::Derivative.

#Curvature Tensors
R_{\mu\nu\rho\sigma}::RiemannTensor.
R_{\mu\nu}::Symmetric.
R::Symbol.

#Metric Upto 4th Order in Coordinates
gmetric := \eta_{\mu\nu}
        - \frac{1}{3} R_{\mu\alpha\nu\beta} y^\alpha y^\beta
        - \frac{1}{6} \nabla_{\gamma}{R_{\mu\alpha\nu\beta}} y^\alpha y^\beta y^\gamma
        + \frac{1}{180} 
        (
        8 R_{\mu\alpha\beta\kappa} g^{\kappa\theta}R_{\theta\gamma\chi\nu}
        - 9 \nabla_{\gamma}{\nabla_{\chi}{R_{\mu\alpha\nu\beta}}}
        )
        y^\alpha y^\beta y^\gamma y^\chi;

#Metric Determinant (Measure) Upto 4th Order in Coordinates
sqrtg := 1
        - \frac{1}{6} R_{\mu\nu} y^\mu y^\nu
        - \frac{1}{12} \nabla_{\rho}{R_{\mu\nu}} y^\mu y^\nu y^\rho
        - 
        (
        \frac{1}{40} \nabla_{\rho}{\nabla_{\sigma}{R_{\mu\nu}}}
        + \frac{1}{180} g^{\kappa\phi}R_{\mu\alpha\nu\phi} g^{\alpha\theta}R_{\rho\theta\sigma\kappa}
        - \frac{1}{72} R_{\mu\nu} R_{\rho\sigma}
        )
        y^\mu y^\nu y^\rho y^\sigma;

#Truncating Machine
def trunc(ex, order):
    extrunc := 0.
    for k in range(order + 1):
        tmp := @(ex).
        keep_weight(tmp, $order=@(k)$)
        extrunc += tmp
    return extrunc

#Definite Integration Machine [Works for upto 8th degree polynomial]  
def int(variable, ex, lower, upper):
    result := 0.
    current := @(ex).
    for k in range(9):
        term := @(current).
        substitute(term, $@(variable) -> @(lower)$)
        if k == 0:
            result := @(result) + @(term) (@(upper)-@(lower)).
        elif k == 1:
            result := @(result) + \frac{1}{2} @(term) (@(upper)-@(lower))**2.
        elif k == 2:
            result := @(result) + \frac{1}{6} @(term) (@(upper)-@(lower))**3.
        elif k == 3:
            result := @(result) + \frac{1}{24} @(term) (@(upper)-@(lower))**4.
        elif k == 4:
            result := @(result) + \frac{1}{120} @(term) (@(upper)-@(lower))**5.
        elif k == 5:
            result := @(result) + \frac{1}{720} @(term) (@(upper)-@(lower))**6.
        elif k == 6:
            result := @(result) + \frac{1}{5040} @(term) (@(upper)-@(lower))**7.
        elif k == 7:
            result := @(result) + \frac{1}{40320} @(term) (@(upper)-@(lower))**8.
        elif k == 8:
            result := @(result) + \frac{1}{362880} @(term) (@(upper)-@(lower))**9.
        current = diff($@(current)$, $@(variable)$)
    return result

def r_piece(ex, p):
    tmp := @(ex).
    for k in range(p):
        tmp = diff($@(tmp)$, $r$)
    substitute(tmp, $r -> 0$)
    if p == 2:
        tmp := \frac{1}{2} @(tmp).
    elif p == 3:
        tmp := \frac{1}{6} @(tmp).
    elif p == 4:
        tmp := \frac{1}{24} @(tmp).
    return tmp
    
def u_piece(ex, p):
    tmp := @(ex).
    for k in range(p):
        tmp = diff($@(tmp)$, $u$)
    substitute(tmp, $u -> 0$)
    if p == 2:
        tmp := \frac{1}{2} @(tmp).
    elif p == 3:
        tmp := \frac{1}{6} @(tmp).
    elif p == 4:
        tmp := \frac{1}{24} @(tmp).
    return tmp
    
def v_piece(ex, p):
    tmp := @(ex).
    for k in range(p):
        tmp = diff($@(tmp)$, $v$)
    substitute(tmp, $v -> 0$)
    if p == 2:
        tmp := \frac{1}{2} @(tmp).
    elif p == 3:
        tmp := \frac{1}{6} @(tmp).
    elif p == 4:
        tmp := \frac{1}{24} @(tmp).
    return tmp
    
def divide_ud(ex):
    for term_node in ex.top().terms():
        term = term_node.ex()
        term := @(term) u**(-d).
        canonicalise(term)
        collect_factors(term)
        term := u @(term).
        canonicalise(term)
        collect_factors(term)
        term_node.replace(term)
    return ex
    
def divide_vd(ex):
    for term_node in ex.top().terms():
        term = term_node.ex()
        term := @(term) v**(-d).
        canonicalise(term)
        collect_factors(term)
        term := v @(term).
        canonicalise(term)
        collect_factors(term)
        term_node.replace(term)
    return ex

def keep_n_count(ex, N):
    out := @(ex).
    for term_node in out.top().terms():
        term = term_node.ex()
        number = len(list(term[r'n']))
        if number != N:
            term_node.erase()
    return out
    
def curvature_simplify_step(ex):
    expand_power(ex)
    distribute(ex)
    canonicalise(ex)
    rename_dummies(ex)
    collect_factors(ex)
    sort_product(ex)
    simplify(ex)
    collect_terms(ex)
    sort_sum(ex)
    
def simplify_all(ex):
    for k in range(3):
        expand_power(ex)
        distribute(ex)
        canonicalise(ex)
        rename_dummies(ex)
        collect_factors(ex)
        sort_product(ex)
        collect_terms(ex)
        sort_sum(ex)
    return ex
    
def rational_simplify(ex):
    map_sympy(ex, "together")
    map_sympy(ex, "factor")
    map_sympy(ex, "cancel")
    map_sympy(ex, "factor")

${}\eta_{\mu \nu} - \frac{1}{3}\,R_{\mu \alpha \nu \beta} y^{\alpha} y^{\beta} - \frac{1}{6}\,\nabla_{\gamma}{R_{\mu \alpha \nu \beta}} y^{\alpha} y^{\beta} y^{\gamma}+\frac{1}{180}\,\left(8\,R_{\mu \alpha \beta \kappa} g^{\kappa \theta} R_{\theta \gamma \chi \nu}-9\,\nabla_{\gamma}\left(\nabla_{\chi}{R_{\mu \alpha \nu \beta}}\right)\right) y^{\alpha} y^{\beta} y^{\gamma} y^{\chi}$

${}1 - \frac{1}{6}\,R_{\mu \nu} y^{\mu} y^{\nu} - \frac{1}{12}\,\nabla_{\rho}{R_{\mu \nu}} y^{\mu} y^{\nu} y^{\rho}-\,\left(\frac{1}{40}\,\nabla_{\rho}\left(\nabla_{\sigma}{R_{\mu \nu}}\right)+\frac{1}{180}\,g^{\kappa \phi} R_{\mu \alpha \nu \phi} g^{\alpha \theta} R_{\rho \theta \sigma \kappa} - \frac{1}{72}\,R_{\mu \nu} R_{\rho \sigma}\right) y^{\mu} y^{\nu} y^{\rho} y^{\sigma}$

# 2# Synge's Function

In [2]:
#Affine parameter
s::Coordinate.

#Endpoint separation
Delta := x^\mu - y^\mu;

#Leading flat geodesic
z1 := y^\mu + s \Delta^\mu;
z1dot = diff(z1, $s$)
z1dot := @(z1dot);

#First non-trivial geodesic correction
z3 := \frac{1}{3} g^{\mu \chi} R_{\chi\alpha\beta\nu} \Delta^\alpha \Delta^\beta y^\nu s (s-1);
z3dot = diff(z3, $s$)
z3dot := @(z3dot);

${}x^{\mu}-\,y^{\mu}$

${}y^{\mu}+s \Delta^{\mu}$

${}\Delta^{\mu}$

${}\frac{1}{3}\,g^{\mu \chi} R_{\chi \alpha \beta \nu} \Delta^{\alpha} \Delta^{\beta} y^{\nu} s \left(s-\,1\right)$

${}\frac{1}{3}\,y^{\nu} g^{\mu \chi} R_{\chi \alpha \beta \nu} \Delta^{\beta} \Delta^{\alpha} s+\frac{1}{3}\,y^{\nu} g^{\mu \chi} R_{\chi \alpha \beta \nu} \Delta^{\beta} \Delta^{\alpha} \left(s-\,1\right)$

In [3]:
#Splitting Metric by Order
g2 := @(gmetric).
keep_weight(g2, $order=2$);
g3 := @(gmetric).
keep_weight(g3, $order=3$);
g4 := @(gmetric).
keep_weight(g4, $order=4$);

#Derivative of 2nd Order of Metric
g2copy := @(g2).
dg2 := \partial_\rho{@(g2copy)}.
product_rule(dg2)
substitute(dg2, $\partial_\rho{y^\alpha} -> \delta^\alpha_\rho$)
unwrap(dg2)
eliminate_kronecker(dg2)
collect_factors(dg2)
sort_product(dg2)
collect_terms(dg2)
sort_sum(dg2);

${} - \frac{1}{3}\,R_{\mu \alpha \nu \beta} y^{\alpha} y^{\beta}$

${} - \frac{1}{6}\,\nabla_{\gamma}{R_{\mu \alpha \nu \beta}} y^{\alpha} y^{\beta} y^{\gamma}$

${}\frac{1}{180}\,\left(8\,R_{\mu \alpha \beta \kappa} g^{\kappa \theta} R_{\theta \gamma \chi \nu}-9\,\nabla_{\gamma}\left(\nabla_{\chi}{R_{\mu \alpha \nu \beta}}\right)\right) y^{\alpha} y^{\beta} y^{\gamma} y^{\chi}$

${} - \frac{1}{3}\,R_{\mu \alpha \nu \rho} y^{\alpha} - \frac{1}{3}\,R_{\mu \rho \nu \beta} y^{\beta}$

In [4]:
#Synge Integrand
SyngeIntegrand := \eta_{\mu\nu}\Delta^\mu\Delta^\nu
                + g2_{\mu\nu}\Delta^\mu\Delta^\nu
                + g3_{\mu\nu}\Delta^\mu\Delta^\nu
                + g4_{\mu\nu}\Delta^\mu\Delta^\nu
                + dg2_{\rho\mu\nu}z3^\rho\Delta^\mu\Delta^\nu
                + 2 g2_{\mu\nu}\Delta^\mu\dot{z3}^\nu
                + \eta_{\mu\nu}\dot{z3}^\mu\dot{z3}^\nu;

${}\eta_{\mu \nu} \Delta^{\mu} \Delta^{\nu}+{g_{2}}_{\mu \nu} \Delta^{\mu} \Delta^{\nu}+{g_{3}}_{\mu \nu} \Delta^{\mu} \Delta^{\nu}+{g_{4}}_{\mu \nu} \Delta^{\mu} \Delta^{\nu}+{dg_{2}}_{\rho \mu \nu} {z_{3}}^{\rho} \Delta^{\mu} \Delta^{\nu}+2\,{g_{2}}_{\mu \nu} \Delta^{\mu} \dot{{z_{3}}}\,^{\nu}+\eta_{\mu \nu} \dot{{z_{3}}}\,^{\mu} \dot{{z_{3}}}\,^{\nu}$

In [5]:
#Symplifying Further
IntegrandCopy := @(SyngeIntegrand).

#Synge Integrand after Substitution
substitute(IntegrandCopy, $g2_{\mu\nu} -> @(g2)$)
substitute(IntegrandCopy, $g3_{\mu\nu} -> @(g3)$)
substitute(IntegrandCopy, $g4_{\mu\nu} -> @(g4)$)
substitute(IntegrandCopy, $dg2_{\rho\mu\nu} -> @(dg2)$)
substitute(IntegrandCopy, $y^\mu -> @(z1)$)
substitute(IntegrandCopy, $z3^\mu -> @(z3)$)
substitute(IntegrandCopy, $\dot{z3}^\mu -> @(z3dot)$)

#Synge simplification
distribute(IntegrandCopy)
canonicalise(IntegrandCopy)
collect_factors(IntegrandCopy)
sort_product(IntegrandCopy)
rename_dummies(IntegrandCopy)
collect_terms(IntegrandCopy)
sort_sum(IntegrandCopy)
canonicalise(IntegrandCopy)

IntegrandCopy;

${}\Delta^{\mu} \Delta^{\nu} \eta_{\mu \nu} - \frac{1}{3}\,R_{\mu \nu \rho \sigma} \Delta^{\mu} \Delta^{\rho} y^{\nu} y^{\sigma} - \frac{1}{6}\,\Delta^{\alpha} \Delta^{\mu} \nabla_{\nu}{R_{\alpha \rho \mu \sigma}} y^{\nu} y^{\rho} y^{\sigma} - \frac{1}{6}\,\Delta^{\alpha} \Delta^{\mu} \Delta^{\nu} \nabla_{\alpha}{R_{\mu \rho \nu \sigma}} s y^{\rho} y^{\sigma} - \frac{1}{20}\,\Delta^{\alpha} \Delta^{\beta} \nabla_{\mu}\left(\nabla_{\nu}{R_{\alpha \rho \beta \sigma}}\right) y^{\mu} y^{\nu} y^{\rho} y^{\sigma} - \frac{1}{20}\,\Delta^{\alpha} \Delta^{\beta} \Delta^{\mu} \Delta^{\nu} \nabla_{\alpha}\left(\nabla_{\beta}{R_{\mu \rho \nu \sigma}}\right) {s}^{2\,} y^{\rho} y^{\sigma} - \frac{1}{20}\,\Delta^{\alpha} \Delta^{\beta} \Delta^{\mu} \nabla_{\nu}\left(\nabla_{\alpha}{R_{\beta \rho \mu \sigma}}\right) s y^{\nu} y^{\rho} y^{\sigma} - \frac{1}{20}\,\Delta^{\alpha} \Delta^{\beta} \Delta^{\mu} \nabla_{\alpha}\left(\nabla_{\nu}{R_{\beta \rho \mu \sigma}}\right) s y^{\nu} y^{\rho} y^{\sigma} - \frac{2}{9}\,R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \Delta^{\alpha} \Delta^{\gamma} \Delta^{\mu} g^{\beta \rho} y^{\kappa} y^{\nu} y^{\sigma}+\frac{2}{45}\,R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \Delta^{\alpha} \Delta^{\mu} g^{\gamma \rho} y^{\beta} y^{\kappa} y^{\nu} y^{\sigma}+\frac{16}{45}\,R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \Delta^{\alpha} \Delta^{\gamma} \Delta^{\mu} g^{\beta \rho} s y^{\kappa} y^{\nu} y^{\sigma} - \frac{8}{45}\,R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \Delta^{\alpha} \Delta^{\gamma} \Delta^{\mu} \Delta^{\rho} {s}^{2\,} g^{\beta \nu} y^{\kappa} y^{\sigma}+\frac{1}{9}\,R_{\alpha \beta \chi \gamma} R_{\kappa \mu \nu \rho} \Delta^{\alpha} \Delta^{\chi} \Delta^{\kappa} \Delta^{\nu} \eta_{\sigma \theta} g^{\beta \sigma} g^{\mu \theta} y^{\gamma} y^{\rho}+\frac{4}{9}\,R_{\alpha \beta \chi \gamma} R_{\kappa \mu \nu \rho} \Delta^{\alpha} \Delta^{\chi} \Delta^{\kappa} \Delta^{\nu} \eta_{\sigma \theta} {s}^{2\,} g^{\beta \sigma} g^{\mu \theta} y^{\gamma} y^{\rho} - \frac{4}{9}\,R_{\alpha \beta \chi \gamma} R_{\kappa \mu \nu \rho} \Delta^{\alpha} \Delta^{\chi} \Delta^{\kappa} \Delta^{\nu} \eta_{\sigma \theta} g^{\beta \sigma} g^{\mu \theta} s y^{\gamma} y^{\rho}$

In [6]:
#Perfomring Integration
SyngeIntegrated = int($s$, IntegrandCopy, 0, 1)

#Simplification
substitute(SyngeIntegrated, $\Delta^\mu -> @(Delta)$)

distribute(SyngeIntegrated)
canonicalise(SyngeIntegrated)
rename_dummies(SyngeIntegrated)
collect_factors(SyngeIntegrated)
sort_product(SyngeIntegrated)
collect_terms(SyngeIntegrated)
sort_sum(SyngeIntegrated)
substitute(SyngeIntegrated, $\eta_{\mu\nu} x^\mu x^\nu -> -t**2 + r**2$)
substitute(SyngeIntegrated, $\eta_{\mu\nu} y^\mu y^\nu -> -\tau**2$)
substitute(SyngeIntegrated, $g^{\mu\alpha} -> \eta^{\mu\alpha}$)
EtaRule := \eta_{\kappa\theta}\eta^{\alpha\theta}\eta^{\mu\kappa} -> \eta^{\mu\alpha}.
XRule := x^\mu -> t e^\mu + r n^\mu.
YRule := y^\mu -> - \tau e^\mu.
substitute(SyngeIntegrated, EtaRule)
substitute(SyngeIntegrated, XRule)
substitute(SyngeIntegrated, YRule)
distribute(SyngeIntegrated)
eliminate_kronecker(SyngeIntegrated)
substitute(SyngeIntegrated, $\eta_{\mu \nu} e^\mu e^\nu -> -1$)
canonicalise(SyngeIntegrated)
rename_dummies(SyngeIntegrated)
collect_factors(SyngeIntegrated)
sort_product(SyngeIntegrated)
collect_terms(SyngeIntegrated)
rename_dummies(SyngeIntegrated)
sort_sum(SyngeIntegrated)
substitute(SyngeIntegrated, $\eta_{\mu\nu}e^\nu n^\mu -> 0$)
substitute(SyngeIntegrated, $\eta_{\chi\theta}\eta^{\mu\theta}\eta^{\alpha\chi} -> \eta^{\alpha\mu}$)
SyngeIntegrated;
print("Number of terms in SyngeIntegrated =", len(list(SyngeIntegrated.top().terms())))

${}-\,{\tau}^{2\,}+{r}^{2\,}-\,{t}^{2\,}-2\,\tau t - \frac{1}{3}\,R_{\mu \nu \rho \sigma} {\tau}^{2\,} {r}^{2\,} e^{\nu} e^{\sigma} n^{\mu} n^{\rho}+\frac{1}{6}\,\nabla_{\alpha}{R_{\mu \nu \rho \sigma}} {\tau}^{3\,} {r}^{2\,} e^{\alpha} e^{\nu} e^{\sigma} n^{\mu} n^{\rho} - \frac{1}{20}\,\nabla_{\alpha}\left(\nabla_{\beta}{R_{\mu \nu \rho \sigma}}\right) {\tau}^{4\,} {r}^{2\,} e^{\alpha} e^{\beta} e^{\nu} e^{\sigma} n^{\mu} n^{\rho} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{4\,} e^{\sigma} e^{\kappa} n^{\nu} n^{\rho} n^{\beta} n^{\gamma} - \frac{4}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{3\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\kappa} n^{\rho} n^{\beta} n^{\gamma} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{4\,} {r}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{2\,} {t}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma}+\frac{2}{15}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\gamma} n^{\rho} n^{\beta} n^{\kappa} t - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\kappa} n^{\rho} n^{\beta} n^{\gamma} t+\frac{1}{9}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\sigma} e^{\beta} e^{\kappa} n^{\nu} n^{\rho} n^{\gamma} t - \frac{4}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{3\,} {r}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma} t$

Number of terms in SyngeIntegrated = 15


# 3# Finding Regions of Integration

###  1# Finding $r_{out}$

In [7]:
#Splitting by Order
FoutCopy := @(SyngeIntegrated).
Fout2 := @(FoutCopy).
keep_weight(Fout2, $order=2$);
Fout4 := @(FoutCopy).
keep_weight(Fout4, $order=4$);
Fout5 := @(FoutCopy).
keep_weight(Fout5, $order=5$);
Fout6 := @(FoutCopy).
keep_weight(Fout6, $order=6$);
rout1 := t + \tau.

${}-\,{\tau}^{2\,}+{r}^{2\,}-\,{t}^{2\,}-2\,\tau t$

${} - \frac{1}{3}\,R_{\mu \nu \rho \sigma} {\tau}^{2\,} {r}^{2\,} e^{\nu} e^{\sigma} n^{\mu} n^{\rho}$

${}\frac{1}{6}\,\nabla_{\alpha}{R_{\mu \nu \rho \sigma}} {\tau}^{3\,} {r}^{2\,} e^{\alpha} e^{\nu} e^{\sigma} n^{\mu} n^{\rho}$

${} - \frac{1}{20}\,\nabla_{\alpha}\left(\nabla_{\beta}{R_{\mu \nu \rho \sigma}}\right) {\tau}^{4\,} {r}^{2\,} e^{\alpha} e^{\beta} e^{\nu} e^{\sigma} n^{\mu} n^{\rho} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{4\,} e^{\sigma} e^{\kappa} n^{\nu} n^{\rho} n^{\beta} n^{\gamma} - \frac{4}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{3\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\kappa} n^{\rho} n^{\beta} n^{\gamma} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{4\,} {r}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma} - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{2\,} {t}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma}+\frac{2}{15}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\gamma} n^{\rho} n^{\beta} n^{\kappa} t - \frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\nu} e^{\sigma} e^{\kappa} n^{\rho} n^{\beta} n^{\gamma} t+\frac{1}{9}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {r}^{3\,} e^{\sigma} e^{\beta} e^{\kappa} n^{\nu} n^{\rho} n^{\gamma} t - \frac{4}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{3\,} {r}^{2\,} e^{\nu} e^{\sigma} e^{\beta} e^{\kappa} n^{\rho} n^{\gamma} t$

In [8]:
Fout2r = diff($@(Fout2)$, $r$)
Fout2rr = diff($@(Fout2r)$, $r$)
Fout4r = diff($@(Fout4)$, $r$)
drout3 := -\frac{@(Fout4)}{@(Fout2r)}.
drout4 := -\frac{@(Fout5)}{@(Fout2r)}.
drout5 := -\frac{@(Fout6) + @(Fout4r) @(drout3) + \frac{1}{2} @(Fout2rr) @(drout3) @(drout3)}{@(Fout2r)}.
substitute(drout3, $r -> @(rout1)$)
substitute(drout4, $r -> @(rout1)$)
substitute(drout5, $r -> @(rout1)$)
rout := @(rout1) + @(drout3) + @(drout4) + @(drout5).
distribute(rout)
canonicalise(rout)
rename_dummies(rout)
collect_factors(rout)
sort_product(rout)
collect_terms(rout)
rename_dummies(rout)
sort_sum(rout);

${}\frac{1}{6}\,R_{\mu \nu \rho \sigma} {\tau}^{2\,} \left(\tau+t\right) e^{\mu} e^{\rho} n^{\nu} n^{\sigma} - \frac{1}{12}\,\nabla_{\alpha}{R_{\mu \nu \rho \sigma}} {\tau}^{3\,} \left(\tau+t\right) e^{\alpha} e^{\mu} e^{\rho} n^{\nu} n^{\sigma}+\frac{1}{40}\,\nabla_{\alpha}\left(\nabla_{\beta}{R_{\mu \nu \rho \sigma}}\right) {\tau}^{4\,} \left(\tau+t\right) e^{\alpha} e^{\beta} e^{\mu} e^{\rho} n^{\nu} n^{\sigma}+\frac{2}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\left(\tau+t\right)}^{2\,} {\tau}^{3\,} e^{\rho} e^{\beta} e^{\gamma} n^{\nu} n^{\sigma} n^{\kappa}+\frac{1}{90}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\left(\tau+t\right)}^{3\,} {\tau}^{2\,} e^{\rho} e^{\gamma} n^{\nu} n^{\sigma} n^{\beta} n^{\kappa}+\frac{1}{90}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{4\,} \left(\tau+t\right) e^{\nu} e^{\rho} e^{\beta} e^{\gamma} n^{\sigma} n^{\kappa}+\frac{1}{18}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} {\tau}^{5\,} e^{\mu} e^{\rho} e^{\beta} e^{\kappa} n^{\nu} n^{\sigma} n^{\alpha} n^{\gamma}+\frac{1}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\left(\tau+t\right)}^{2\,} {\tau}^{2\,} e^{\rho} e^{\beta} e^{\gamma} n^{\nu} n^{\sigma} n^{\kappa} t+\frac{1}{90}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{2\,} {t}^{2\,} \left(\tau+t\right) e^{\nu} e^{\rho} e^{\beta} e^{\gamma} n^{\sigma} n^{\kappa}+\frac{2}{45}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{3\,} \left(\tau+t\right) e^{\nu} e^{\rho} e^{\beta} e^{\gamma} n^{\sigma} n^{\kappa} t - \frac{1}{72}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} {\tau}^{4\,} \left(\tau+t\right) e^{\mu} e^{\rho} e^{\alpha} e^{\gamma} n^{\nu} n^{\sigma} n^{\beta} n^{\kappa}+\frac{1}{18}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} {\tau}^{4\,} e^{\mu} e^{\rho} e^{\beta} e^{\kappa} n^{\nu} n^{\sigma} n^{\alpha} n^{\gamma} t+\tau+t$

###  2# Finding $t_{out}$

In [9]:
#Splitting by Order
HoutCopy := @(SyngeIntegrated).
substitute(HoutCopy, $r -> -t$)
Hout2 := @(HoutCopy).
keep_weight(Hout2, $order=2$);
Hout4 := @(HoutCopy).
keep_weight(Hout4, $order=4$)
Hout5 := @(HoutCopy).
keep_weight(Hout5, $order=5$)
Hout6 := @(HoutCopy).
keep_weight(Hout6, $order=6$)

tout1 := -\frac{1}{2}\tau;

${}-\,{\tau}^{2\,}-2\,\tau t$

${} - \frac{1}{2}\,\tau$

In [10]:
Hout2t = diff($@(Hout2)$, $t$)
Hout2tt = diff($@(Hout2t)$, $t$)
Hout4t = diff($@(Hout4)$, $t$)
dtout3 := -\frac{@(Hout4)}{@(Hout2t)}.
dtout4 := -\frac{@(Hout5)}{@(Hout2t)}.
dtout5 := -\frac{@(Hout6) + @(Hout4t) @(dtout3) + \frac{1}{2} @(Hout2tt) @(dtout3) @(dtout3)}{@(Hout2t)}.
substitute(dtout3, $t -> @(tout1)$)
substitute(dtout4, $t -> @(tout1)$)
substitute(dtout5, $t -> @(tout1)$)
tout := @(tout1) + @(dtout3) + @(dtout4) + @(dtout5).
distribute(tout)
canonicalise(tout)
rename_dummies(tout)
collect_factors(tout)
sort_product(tout)
collect_terms(tout)
rename_dummies(tout)
sort_sum(tout)
tout;

${} - \frac{1}{24}\,R_{\mu \nu \rho \sigma} {\tau}^{3\,} e^{\mu} e^{\rho} n^{\nu} n^{\sigma}+\frac{1}{48}\,\nabla_{\alpha}{R_{\mu \nu \rho \sigma}} {\tau}^{4\,} e^{\alpha} e^{\mu} e^{\rho} n^{\nu} n^{\sigma} - \frac{1}{160}\,\nabla_{\alpha}\left(\nabla_{\beta}{R_{\mu \nu \rho \sigma}}\right) {\tau}^{5\,} e^{\alpha} e^{\beta} e^{\mu} e^{\rho} n^{\nu} n^{\sigma}+\frac{1}{480}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{5\,} e^{\nu} e^{\rho} e^{\beta} e^{\gamma} n^{\sigma} n^{\kappa} - \frac{1}{240}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{5\,} e^{\rho} e^{\beta} e^{\gamma} n^{\nu} n^{\sigma} n^{\kappa} - \frac{1}{1440}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} \eta^{\alpha \mu} {\tau}^{5\,} e^{\rho} e^{\gamma} n^{\nu} n^{\sigma} n^{\beta} n^{\kappa} - \frac{1}{144}\,R_{\mu \nu \rho \sigma} R_{\alpha \beta \gamma \kappa} {\tau}^{5\,} e^{\mu} e^{\rho} e^{\beta} e^{\kappa} n^{\nu} n^{\sigma} n^{\alpha} n^{\gamma} - \frac{1}{2}\,\tau$

# 4# Performing Integration

### 1# Performing Radial Integration

In [11]:
sqrtgCopy := @(sqrtg).
substitute(sqrtgCopy, $y^\mu -> t e^\mu + r n^\mu$)
distribute(sqrtgCopy)
canonicalise(sqrtgCopy)
rename_dummies(sqrtgCopy)
collect_factors(sqrtgCopy)
collect_terms(sqrtgCopy)
distribute(sqrtgCopy)
sort_sum(sqrtgCopy)

G0 := @(sqrtgCopy).
keep_weight(G0, $order=0$)
G2 := @(sqrtgCopy).
keep_weight(G2, $order=2$)
sort_product(G2)
rename_dummies(G2)
canonicalise(G2)
collect_terms(G2)
G3 := @(sqrtgCopy).
keep_weight(G3, $order=3$)
sort_product(G3)
rename_dummies(G3)
canonicalise(G3)
collect_terms(G3)
G4 := @(sqrtgCopy).
keep_weight(G4, $order=4$)
sort_product(G4)
rename_dummies(G4)
canonicalise(G4)
collect_terms(G4)

G20 = r_piece(G2, 0)
G21 = r_piece(G2, 1)
G22 = r_piece(G2, 2)
G30 = r_piece(G3, 0)
G31 = r_piece(G3, 1)
G32 = r_piece(G3, 2)
G33 = r_piece(G3, 3)
G40 = r_piece(G4, 0)
G41 = r_piece(G4, 1)
G42 = r_piece(G4, 2)
G43 = r_piece(G4, 3)
G44 = r_piece(G4, 4)

routdm1 := (@(rout1))**(d-1)
            + (d-1) (@(rout1))**(d-2) ( @(drout3) + @(drout4) + @(drout5) )
            + \frac{(d-1)*(d-2)}{2} (@(rout1))**(d-3) @(drout3) @(drout3).
routd := (@(rout1))**d
            + d (@(rout1))**(d-1) ( @(drout3) + @(drout4) ).
routdp1 := (@(rout1))**(d+1)
            + (d+1) (@(rout1))**d @(drout3).
routdp2 := (@(rout1))**(d+2).
routdp3 := (@(rout1))**(d+3).

RadialOut0 := \frac{1}{d-1} @(routdm1).
RadialOut2 := @(G20) \frac{1}{d-1} ((@(rout1))**(d-1) + (d-1) (@(rout1))**(d-2) @(drout3))
            + @(G21) \frac{1}{d} ((@(rout1))**d + d (@(rout1))**(d-1) @(drout3))
            + @(G22) \frac{1}{d+1} @(routdp1).
RadialOut3 := @(G30) \frac{1}{d-1} (@(rout1))**(d-1)
            + @(G31) \frac{1}{d} (@(rout1))**d
            + @(G32) \frac{1}{d+1} (@(rout1))**(d+1)
            + @(G33) \frac{1}{d+2} @(routdp2).
RadialOut4 := @(G40) \frac{1}{d-1} (@(rout1))**(d-1)
            + @(G41) \frac{1}{d} (@(rout1))**d
            + @(G42) \frac{1}{d+1} (@(rout1))**(d+1)
            + @(G43) \frac{1}{d+2} (@(rout1))**(d+2)
            + @(G44) \frac{1}{d+3} @(routdp3).
          
RadialPast0 := \frac{1}{d-1} (-t)**(d-1).
RadialPast2 := @(G20) \frac{1}{d-1} (-t)**(d-1)
            + @(G21) \frac{1}{d} (-t)**(d)
            + @(G22) \frac{1}{d+1} (-t)**(d+1).
RadialPast3 := @(G30) \frac{1}{d-1} (-t)**(d-1)
            + @(G31) \frac{1}{d} (-t)**(d)
            + @(G32) \frac{1}{d+1} (-t)**(d+1)
            + @(G33) \frac{1}{d+2} (-t)**(d+2).
RadialPast4 := @(G40) \frac{1}{d-1} (-t)**(d-1)
            + @(G41) \frac{1}{d} (-t)**(d)
            + @(G42) \frac{1}{d+1} (-t)**(d+1)
            + @(G43) \frac{1}{d+2} (-t)**(d+2)
            + @(G44) \frac{1}{d+3} (-t)**(d+3).

RadialOut := @(RadialOut0) + @(RadialOut2) + @(RadialOut3) + @(RadialOut4).
RadialPast := @(RadialPast0) + @(RadialPast2) + @(RadialPast3) + @(RadialPast4).
#RadialOut is the radial-integrated from r=0 to r=rout boundary.
#RadialPast is the radial-integrated from r=0 to r=-t boundary.

### 2# Performing Time Integration

In [12]:
#Simplifying Before Integration
TimeOutCore := @(RadialOut).
substitute(TimeOutCore, $t -> u-\tau$)
expand_power(TimeOutCore)
distribute(TimeOutCore)
divide_ud(TimeOutCore)
canonicalise(TimeOutCore)
rename_dummies(TimeOutCore)
collect_factors(TimeOutCore)
sort_product(TimeOutCore)
collect_terms(TimeOutCore)
sort_sum(TimeOutCore)

T0 := @(TimeOutCore).
keep_weight(T0, $order=0$)
distribute(T0)
sort_product(T0)
rename_dummies(T0)
canonicalise(T0)
collect_factors(T0)
collect_terms(T0)
sort_sum(T0)
T1 := @(TimeOutCore).
keep_weight(T1, $order=1$)
distribute(T1)
sort_product(T1)
rename_dummies(T1)
canonicalise(T1)
collect_factors(T1)
collect_terms(T1)
sort_sum(T1)
T2 := @(TimeOutCore).
keep_weight(T2, $order=2$)
distribute(T2)
sort_product(T2)
rename_dummies(T2)
canonicalise(T2)
collect_factors(T2)
collect_terms(T2)
sort_sum(T2)
T3 := @(TimeOutCore).
keep_weight(T3, $order=3$)
distribute(T3)
sort_product(T3)
rename_dummies(T3)
canonicalise(T3)
collect_factors(T3)
collect_terms(T3)
sort_sum(T3)
T4 := @(TimeOutCore).
keep_weight(T4, $order=4$)
distribute(T4)
sort_product(T4)
rename_dummies(T4)
canonicalise(T4)
collect_factors(T4)
collect_terms(T4)
sort_sum(T4)

C00 = u_piece(T0,0)
C10 = u_piece(T1,0)
C11 = u_piece(T1,1)
C20 = u_piece(T2,0)
C21 = u_piece(T2,1)
C22 = u_piece(T2,2)
C30 = u_piece(T3,0)
C31 = u_piece(T3,1)
C32 = u_piece(T3,2)
C33 = u_piece(T3,3)
C40 = u_piece(T4,0)
C41 = u_piece(T4,1)
C42 = u_piece(T4,2)
C43 = u_piece(T4,3)
C44 = u_piece(T4,4)

uout1 := @(tout1) + \tau.

uoutd := (@(uout1))**d
       + d (@(uout1))**(d-1) ( @(dtout3) + @(dtout4) + @(dtout5) )
       + \frac{d*(d-1)}{2} (@(uout1))**(d-2) @(dtout3) @(dtout3).       
uoutdp1 := (@(uout1))**(d+1)
         + (d+1) (@(uout1))**d ( @(dtout3) + @(dtout4) ).         
uoutdp2 := (@(uout1))**(d+2)
         + (d+2) (@(uout1))**(d+1) @(dtout3).      
uoutdp3 := (@(uout1))**(d+3).
uoutdp4 := (@(uout1))**(d+4).

TimeOut0 := @(C00) \frac{1}{d} @(uoutd).
TimeOut1 := @(C10) \frac{1}{d} ((@(uout1))**d + d (@(uout1))**(d-1) ( @(dtout3) + @(dtout4)))
           + @(C11) \frac{1}{d+1} @(uoutdp1).
TimeOut2 := @(C20) \frac{1}{d} ((@(uout1))**d + d (@(uout1))**(d-1) @(dtout3))
           + @(C21) \frac{1}{d+1} ((@(uout1))**(d+1) + (d+1) (@(uout1))**d @(dtout3))
           + @(C22) \frac{1}{d+2} @(uoutdp2).
TimeOut3 := @(C30) \frac{1}{d} (@(uout1))**d
           + @(C31) \frac{1}{d+1} (@(uout1))**(d+1)
           + @(C32) \frac{1}{d+2} (@(uout1))**(d+2)
           + @(C33) \frac{1}{d+3} @(uoutdp3).
TimeOut4 := @(C40) \frac{1}{d} (@(uout1))**d
           + @(C41) \frac{1}{d+1} (@(uout1))**(d+1)
           + @(C42) \frac{1}{d+2} (@(uout1))**(d+2)
           + @(C43) \frac{1}{d+3} (@(uout1))**(d+3)
           + @(C44) \frac{1}{d+4} @(uoutdp4).

TimeOut := @(TimeOut0) + @(TimeOut1) + @(TimeOut2) + @(TimeOut3) + @(TimeOut4).

#Simplifying Before Integration
TimePastCore := @(RadialPast).
substitute(TimePastCore, $t -> -v$)
expand_power(TimePastCore)
distribute(TimePastCore)
divide_vd(TimePastCore)
canonicalise(TimePastCore)
rename_dummies(TimePastCore)
collect_factors(TimePastCore)
sort_product(TimePastCore)
collect_terms(TimePastCore)
sort_sum(TimePastCore)

P0 := @(TimePastCore).
keep_weight(P0, $order=0$)
distribute(P0)
sort_product(P0)
rename_dummies(P0)
canonicalise(P0)
collect_factors(P0)
collect_terms(P0)
sort_sum(P0)
P1 := @(TimePastCore).
keep_weight(P1, $order=1$)
distribute(P1)
sort_product(P1)
rename_dummies(P1)
canonicalise(P1)
collect_factors(P1)
collect_terms(P1)
sort_sum(P1)
P2 := @(TimePastCore).
keep_weight(P2, $order=2$)
distribute(P2)
sort_product(P2)
rename_dummies(P2)
canonicalise(P2)
collect_factors(P2)
collect_terms(P2)
sort_sum(P2)
P3 := @(TimePastCore).
keep_weight(P3, $order=3$)
distribute(P3)
sort_product(P3)
rename_dummies(P3)
canonicalise(P3)
collect_factors(P3)
collect_terms(P3)
sort_sum(P3)
P4 := @(TimePastCore).
keep_weight(P4, $order=4$)
distribute(P4)
sort_product(P4)
rename_dummies(P4)
canonicalise(P4)
collect_factors(P4)
collect_terms(P4)
sort_sum(P4)

D00 = v_piece(P0,0)
D10 = v_piece(P1,0)
D11 = v_piece(P1,1)
D20 = v_piece(P2,0)
D21 = v_piece(P2,1)
D22 = v_piece(P2,2)
D30 = v_piece(P3,0)
D31 = v_piece(P3,1)
D32 = v_piece(P3,2)
D33 = v_piece(P3,3)
D40 = v_piece(P4,0)
D41 = v_piece(P4,1)
D42 = v_piece(P4,2)
D43 = v_piece(P4,3)
D44 = v_piece(P4,4)

vout1 := - @(tout1).

voutd := (@(vout1))**d
       - d (@(vout1))**(d-1) ( @(dtout3) + @(dtout4) + @(dtout5) )
       + \frac{d*(d-1)}{2} (@(vout1))**(d-2) @(dtout3) @(dtout3).
voutdp1 := (@(vout1))**(d+1)
         - (d+1) (@(vout1))**d ( @(dtout3) + @(dtout4) ).
voutdp2 := (@(vout1))**(d+2)
         - (d+2) (@(vout1))**(d+1) @(dtout3).
voutdp3 := (@(vout1))**(d+3).
voutdp4 := (@(vout1))**(d+4).

TimePast0 := @(D00) \frac{1}{d} @(voutd).
TimePast1 := @(D10) \frac{1}{d} ((@(vout1))**d - d (@(vout1))**(d-1) (@(dtout3) + @(dtout4)))
           + @(D11) \frac{1}{d+1} @(voutdp1).
TimePast2 := @(D20) \frac{1}{d} ((@(vout1))**d - d (@(vout1))**(d-1) @(dtout3))
           + @(D21) \frac{1}{d+1} ((@(vout1))**(d+1) - (d+1) (@(vout1))**d @(dtout3))
           + @(D22) \frac{1}{d+2} @(voutdp2).
TimePast3 := @(D30) \frac{1}{d} (@(vout1))**d
           + @(D31) \frac{1}{d+1} (@(vout1))**(d+1)
           + @(D32) \frac{1}{d+2} (@(vout1))**(d+2)
           + @(D33) \frac{1}{d+3} @(voutdp3).
TimePast4 := @(D40) \frac{1}{d} (@(vout1))**d
           + @(D41) \frac{1}{d+1} (@(vout1))**(d+1)
           + @(D42) \frac{1}{d+2} (@(vout1))**(d+2)
           + @(D43) \frac{1}{d+3} (@(vout1))**(d+3)
           + @(D44) \frac{1}{d+4} @(voutdp4).

TimePast := @(TimePast0) + @(TimePast1) + @(TimePast2) + @(TimePast3) + @(TimePast4).

TimeTotal := @(TimeOut) + @(TimePast).

### 3# Angular Integration

In [13]:
TimeTotal2 := @(TimeTotal).
#Temporarily scaling tau for simplification
substitute(TimeTotal2, $\tau -> 2\tau$)
simplify_all(TimeTotal2)
AngularStart := @(TimeTotal2).
GInverseRule := g^{\mu\nu} -> \eta^{\mu\nu}.
substitute(AngularStart, GInverseRule)

HRule := h^{\mu\nu} -> \eta^{\mu\nu} + e^\mu e^\nu.

N0 = keep_n_count(AngularStart, 0)
N1 = keep_n_count(AngularStart, 1)
N2 = keep_n_count(AngularStart, 2)
N3 = keep_n_count(AngularStart, 3)
N4 = keep_n_count(AngularStart, 4)

AngularRule2 := n^\mu n^\nu -> \frac{1}{d-1} h^{\mu\nu}.
AngularRule4 := n^\mu n^\nu n^\rho n^\sigma ->
                \frac{1}{(d-1)*(d+1)}
                (
                    h^{\mu\nu} h^{\rho\sigma}
                    + h^{\mu\rho} h^{\nu\sigma}
                    + h^{\mu\sigma} h^{\nu\rho}
                ).
                
distribute(N2)
canonicalise(N2)
sort_product(N2)
substitute(N2, AngularRule2)
distribute(N4)
canonicalise(N4)
sort_product(N4)
substitute(N4, AngularRule4)

AngularTotal := @(N0) + @(N2) + @(N4).
AngularTotal := \Omega @(AngularTotal).

substitute(AngularTotal, HRule)
distribute(AngularTotal)
canonicalise(AngularTotal)
rename_dummies(AngularTotal)
collect_factors(AngularTotal)
sort_product(AngularTotal)
collect_terms(AngularTotal)
sort_sum(AngularTotal)

\eta_{\mu\nu}::Symmetric.
\eta^{\mu\nu}::Symmetric.
\eta_{\mu\nu}::Metric.
\eta^{\mu\nu}::InverseMetric.

simplify_all(AngularTotal)
EtaENorm :=\eta_{\mu\nu} e^\mu e^\nu -> -1.
substitute(AngularTotal, EtaENorm)
simplify_all(AngularTotal)

ERule := e^\mu -> -\frac{1}{2} \frac{y^\mu}{\tau}.
substitute(AngularTotal, ERule)
simplify_all(AngularTotal)

# 5# Simplification and Covariantization

In [14]:
VolumeReduced := @(AngularTotal).

RuleRiemannRicci1 := R_{\mu\nu\rho\sigma} \eta^{\mu\rho} -> R_{\nu\sigma}.
RuleRiemannRicci2 := R_{\mu\nu\rho\sigma} \eta^{\nu\sigma} -> R_{\mu\rho}.
RuleRicciScalar := R_{\mu\nu} \eta^{\mu\nu} -> R.

for k in range(3):
    substitute(VolumeReduced, RuleRiemannRicci2)
    curvature_simplify_step(VolumeReduced)

for k in range(3):
    substitute(VolumeReduced, RuleRiemannRicci1)
    curvature_simplify_step(VolumeReduced)
    
for k in range(3):
    substitute(VolumeReduced, RuleRicciScalar)
    curvature_simplify_step(VolumeReduced)

#Taking Omega and tau^d common. [Remember we have still new tau]
VolumeStripped := @(VolumeReduced) \Omega**(-1) \tau**(-d).
expand_power(VolumeStripped)
distribute(VolumeStripped)
sort_product(VolumeStripped)
rename_dummies(VolumeStripped)
canonicalise(VolumeStripped)
collect_factors(VolumeStripped)
collect_terms(VolumeStripped)
sort_sum(VolumeStripped)

#Rules for Simplification
R1_rule  := R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \eta^{\mu \alpha} \eta^{\rho \gamma} y^{\beta} y^{\kappa} y^{\nu} y^{\sigma} -> R1.
R2_rule  := R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \eta^{\mu \alpha} \eta^{\nu \gamma} \eta^{\rho \beta} y^{\kappa} y^{\sigma} \tau**2 -> R2.
R3_rule  := R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \eta^{\mu \alpha} \eta^{\nu \beta} \eta^{\rho \gamma} y^{\kappa} y^{\sigma} \tau**2-> 2 R2.
R4_rule  := R_{\mu \nu} R_{\rho \sigma \alpha \beta} \eta^{\mu \rho} \eta^{\nu \alpha} y^{\beta} y^{\sigma} \tau**2 -> R4.
R5_rule  := R_{\mu \nu} R_{\rho \sigma} y^{\mu} y^{\nu} y^{\rho} y^{\sigma} -> R5.
R6_rule  := R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \eta^{\mu \alpha} \eta^{\nu \gamma} \eta^{\rho \beta} \eta^{\sigma \kappa} \tau**4 -> R6.
R7_rule  := R_{\alpha \beta \gamma \kappa} R_{\mu \nu \rho \sigma} \eta^{\mu \alpha} \eta^{\nu \beta} \eta^{\rho \gamma} \eta^{\sigma \kappa} \tau**4 -> 2 R6.
R8_rule  := R_{\mu \nu} R_{\rho \sigma} \eta^{\mu \rho} y^{\nu} y^{\sigma} \tau**2 -> R8.
R9_rule  := R R_{\mu \nu} y^{\mu} y^{\nu} \tau**2 -> R9.
R10_rule := \nabla_{\rho}{\nabla_{\sigma}{R_{\mu \nu}}} y^{\mu} y^{\nu} y^{\rho} y^{\sigma} -> R10.
R11_rule := R_{\mu \nu} R_{\rho \sigma} \eta^{\mu \rho} \eta^{\nu \sigma} \tau**4 -> R11.
R12_rule := \nabla_{\rho}{R_{\mu \nu}} y^{\mu} y^{\nu} y^{\rho} -> R12.
R13_rule := R_{\mu \nu} y^{\mu} y^{\nu} -> R13.
R14_rule := R**2 \tau**4 -> R14.
R15_rule := R \tau**2 -> R15.

VolumeSeparated := @(VolumeStripped).
substitute(VolumeSeparated, R1_rule)
substitute(VolumeSeparated, R2_rule)
substitute(VolumeSeparated, R3_rule)
substitute(VolumeSeparated, R4_rule)
substitute(VolumeSeparated, R5_rule)
substitute(VolumeSeparated, R6_rule)
substitute(VolumeSeparated, R7_rule)
substitute(VolumeSeparated, R8_rule)
substitute(VolumeSeparated, R9_rule)
substitute(VolumeSeparated, R10_rule)
substitute(VolumeSeparated, R11_rule)
substitute(VolumeSeparated, R12_rule)
substitute(VolumeSeparated, R13_rule)
substitute(VolumeSeparated, R14_rule)
substitute(VolumeSeparated, R15_rule)

T0 := @(VolumeSeparated).
substitute(T0, $R1 -> 0$)
substitute(T0, $R2 -> 0$)
substitute(T0, $R3 -> 0$)
substitute(T0, $R4 -> 0$)
substitute(T0, $R5 -> 0$)
substitute(T0, $R6 -> 0$)
substitute(T0, $R7 -> 0$)
substitute(T0, $R8 -> 0$)
substitute(T0, $R9 -> 0$)
substitute(T0, $R10 -> 0$)
substitute(T0, $R11 -> 0$)
substitute(T0, $R12 -> 0$)
substitute(T0, $R13 -> 0$)
substitute(T0, $R14 -> 0$)
substitute(T0, $R15 -> 0$)

T1 = diff(VolumeSeparated, $R1$)
T2 = diff(VolumeSeparated, $R2$)
T3 = diff(VolumeSeparated, $R3$)
T4 = diff(VolumeSeparated, $R4$)
T5 = diff(VolumeSeparated, $R5$)
T6 = diff(VolumeSeparated, $R6$)
T7 = diff(VolumeSeparated, $R7$)
T8 = diff(VolumeSeparated, $R8$)
T9 = diff(VolumeSeparated, $R9$)
T10 = diff(VolumeSeparated, $R10$)
T11 = diff(VolumeSeparated, $R11$)
T12 = diff(VolumeSeparated, $R12$)
T13 = diff(VolumeSeparated, $R13$)
T14 = diff(VolumeSeparated, $R14$)
T15 = diff(VolumeSeparated, $R15$)

rational_simplify(T0)
rational_simplify(T1)
rational_simplify(T2)
rational_simplify(T3)
rational_simplify(T4)
rational_simplify(T5)
rational_simplify(T6)
rational_simplify(T7)
rational_simplify(T8)
rational_simplify(T9)
rational_simplify(T10)
rational_simplify(T11)
rational_simplify(T12)
rational_simplify(T13)
rational_simplify(T14)
rational_simplify(T15)

VolumeCollected := @(T0) + @(T1) R1 + @(T2) R2 + @(T3) R3 + @(T4) R4 + @(T5) R5 + @(T6) R6 + @(T7) R7 + @(T8) R8 + @(T9) R9 + @(T10) R10 + @(T11) R11 + @(T12) R12 + @(T13) R13 + @(T14) R14 + @(T15) R15.
collect_terms(VolumeCollected)
sort_sum(VolumeCollected)

reverseR1_rule := R1 -> R_{\alpha\beta\gamma\kappa} R^{\alpha}{}_{\nu}{}^{\gamma}{}_{\sigma} y^{\beta} y^{\kappa} y^{\nu} y^{\sigma}.
reverseR2_rule := R2 -> R_{\alpha\beta\gamma\kappa} R^{\alpha\gamma\beta}{}_{\sigma} y^{\kappa} y^{\sigma} \tau**2.
#reverseR3_rule := R3 -> R_{\alpha\beta\gamma\kappa} R^{\alpha\beta\gamma}{}_{\sigma} y^{\kappa} y^{\sigma} \tau**2.
reverseR4_rule := R4 -> R_{\mu\nu} R^{\mu}{}_{\sigma}{}^{\nu}{}_{\beta} y^{\beta} y^{\sigma} \tau**2.
reverseR5_rule := R5 -> R_{\mu\nu} R_{\rho\sigma} y^{\mu} y^{\nu} y^{\rho} y^{\sigma}.
reverseR6_rule := R6 -> R_{\alpha\beta\gamma\kappa} R^{\alpha\gamma\beta\kappa} \tau**4.
#reverseR7_rule := R7 -> R_{\alpha\beta\gamma\kappa} R^{\alpha\beta\gamma\kappa} \tau**4.
reverseR8_rule := R8 -> R_{\mu\nu} R^{\mu}{}_{\sigma} y^{\nu} y^{\sigma} \tau**2.
reverseR9_rule := R9 -> R R_{\mu\nu} y^{\mu} y^{\nu} \tau**2.
reverseR10_rule := R10 -> \nabla_{\rho}{\nabla_{\sigma}{R_{\mu\nu}}} y^{\mu} y^{\nu} y^{\rho} y^{\sigma}.
reverseR11_rule := R11 -> R_{\mu\nu} R^{\mu\nu} \tau**4.
reverseR12_rule := R12 -> \nabla_{\rho}{R_{\mu\nu}} y^{\mu} y^{\nu} y^{\rho}.
reverseR13_rule := R13 -> R_{\mu\nu} y^{\mu} y^{\nu}.
reverseR14_rule := R14 -> R**2 \tau**4.
reverseR15_rule := R15 -> R \tau**2.

VolumeReconstructed := @(VolumeCollected).
substitute(VolumeReconstructed, reverseR1_rule)
substitute(VolumeReconstructed, reverseR2_rule)
#substitute(VolumeReconstructed, reverseR3_rule)
substitute(VolumeReconstructed, reverseR4_rule)
substitute(VolumeReconstructed, reverseR5_rule)
substitute(VolumeReconstructed, reverseR6_rule)
#substitute(VolumeReconstructed, reverseR7_rule)
substitute(VolumeReconstructed, reverseR8_rule)
substitute(VolumeReconstructed, reverseR9_rule)
substitute(VolumeReconstructed, reverseR10_rule)
substitute(VolumeReconstructed, reverseR11_rule)
substitute(VolumeReconstructed, reverseR12_rule)
substitute(VolumeReconstructed, reverseR13_rule)
substitute(VolumeReconstructed, reverseR14_rule)
substitute(VolumeReconstructed, reverseR15_rule)

VolumeFinal := @(VolumeReconstructed) \Omega \tau**d.
substitute(VolumeFinal, $\tau -> \frac{1}{2}\tau$)
collect_factors(VolumeFinal)
sort_product(VolumeFinal)
collect_terms(VolumeFinal)
sort_sum(VolumeFinal)
VolumeFinal;

${}\Omega {\left(\frac{1}{2}\,\tau\right)}^{d} \left(2\,{\left(\left(-\,1+d\right) d\right)}^{-\,1} - \frac{1}{12}\,R {\left(\left(-\,1+d\right) \left(1+d\right) \left(2\,+d\right)\right)}^{-\,1} {\tau}^{2\,}+\frac{1}{576}\,{R}^{2\,} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right) \left(4\,+d\right)\right)}^{-\,1} {\tau}^{4\,}+\frac{1}{360}\,R_{\mu \nu} R^{\mu \nu} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right) \left(4\,+d\right)\right)}^{-\,1} {\tau}^{4\,}+\frac{1}{12}\,R_{\mu \nu} {\left(\left(-\,1+d\right) \left(1+d\right)\right)}^{-\,1} y^{\mu} y^{\nu} - \frac{1}{480}\,R_{\alpha \beta \gamma \kappa} R^{\alpha \gamma \beta \kappa} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right) \left(4\,+d\right)\right)}^{-\,1} {\tau}^{4\,} - \frac{1}{12}\,\nabla_{\rho}{R_{\mu \nu}} {\left(\left(-\,1+d\right) d\right)}^{-\,1} y^{\mu} y^{\nu} y^{\rho} - \frac{1}{288}\,R R_{\mu \nu} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right)\right)}^{-\,1} {\tau}^{2\,} y^{\mu} y^{\nu}+\frac{1}{240}\,R_{\alpha \beta \gamma \kappa} R^{\alpha \gamma \beta}\,_{\sigma} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right)\right)}^{-\,1} {\tau}^{2\,} y^{\kappa} y^{\sigma} - \frac{1}{12}\,\eta^{\alpha \mu} \nabla_{\nu}{R_{\alpha \rho \mu \sigma}} {\left({\left(-\,1+d\right)}^{2\,} d\right)}^{-\,1} y^{\nu} y^{\rho} y^{\sigma}+\frac{1}{12}\,\eta^{\alpha \mu} \nabla_{\nu}{R_{\alpha \rho \mu \sigma}} {\left(-\,1+d\right)}^{-2\,} y^{\nu} y^{\rho} y^{\sigma} - \frac{1}{40}\,\nabla_{\rho}\left(\nabla_{\sigma}{R_{\mu \nu}}\right) {\left(\left(-\,1+d\right) d\right)}^{-\,1} y^{\mu} y^{\nu} y^{\rho} y^{\sigma}+\frac{1}{720}\,R_{\mu \nu} R^{\mu}\,_{\sigma} {\left(\left(-\,1+d\right) \left(1+d\right) \left(2\,+d\right) \left(3\,+d\right)\right)}^{-\,1} {\tau}^{2\,} \left(26\,+7\,d\right) y^{\nu} y^{\sigma} - \frac{1}{720}\,R_{\mu \nu} R^{\mu}\,_{\sigma}\,^{\nu}\,_{\beta} {\left(\left(-\,1+d\right) \left(1+d\right) \left(2\,+d\right) \left(3\,+d\right)\right)}^{-\,1} {\tau}^{2\,} \left(34\,+11\,d\right) y^{\beta} y^{\sigma} - \frac{1}{40}\,\eta^{\alpha \beta} \nabla_{\mu}\left(\nabla_{\nu}{R_{\alpha \rho \beta \sigma}}\right) {\left({\left(-\,1+d\right)}^{2\,} d\right)}^{-\,1} y^{\mu} y^{\nu} y^{\rho} y^{\sigma}+\frac{1}{40}\,\eta^{\alpha \beta} \nabla_{\mu}\left(\nabla_{\nu}{R_{\alpha \rho \beta \sigma}}\right) {\left(-\,1+d\right)}^{-2\,} y^{\mu} y^{\nu} y^{\rho} y^{\sigma}+\frac{1}{576}\,R_{\mu \nu} R_{\rho \sigma} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right)\right)}^{-\,1} \left(2\,+d\right) y^{\mu} y^{\nu} y^{\rho} y^{\sigma}+\frac{1}{1440}\,R_{\alpha \beta \gamma \kappa} R^{\alpha}\,_{\nu}\,^{\gamma}\,_{\sigma} {\left(\left(-\,1+d\right) \left(1+d\right) \left(3\,+d\right)\right)}^{-\,1} \left(22\,+7\,d\right) y^{\beta} y^{\kappa} y^{\nu} y^{\sigma}\right)$